<a href="https://colab.research.google.com/github/alkhadrifuad/DataScience_250401020114_AlkhadriFuad/blob/master/Pertemuan12_alkhadrifuad_250401020114.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Generate & Eksplorasi Dataset Transaksi

In [2]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
produk = ['Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
 'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega']

# Buat 50 transaksi, tiap transaksi berisi 2-5 produk
transaksi = []
for _ in range(50):
 n_item = np.random.randint(2, 6)
 transaksi.append(list(np.random.choice(produk, n_item, replace=False)))
# Suntikkan pola: Roti sering bersama Selai
for i in range(0, 20):
 if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
  transaksi[i].append('Selai')
display('Contoh transaksi:', transaksi[:3])
display('Jumlah transaksi:', len(transaksi))

'Contoh transaksi:'

[[np.str_('Keju'),
  np.str_('Roti'),
  np.str_('Mentega'),
  np.str_('Kopi'),
  'Selai'],
 [np.str_('Roti'),
  np.str_('Kopi'),
  np.str_('Teh'),
  np.str_('Selai'),
  np.str_('Mentega')],
 [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]

'Jumlah transaksi:'

50

In [3]:
from collections import Counter

# Gabungkan semua produk dari semua transaksi ke dalam satu list
all_products = [item for sublist in transaksi for item in sublist]

# Hitung frekuensi masing-masing produk
product_frequency = Counter(all_products)

# Ubah ke pandas Series untuk tampilan yang lebih baik
product_frequency_df = pd.Series(product_frequency).sort_values(ascending=False)

display('Frekuensi Setiap Produk:', product_frequency_df)

'Frekuensi Setiap Produk:'

,0
Selai,26
Teh,23
Mentega,21
Telur,18
Keju,17
Roti,16
Kopi,16
Susu,16
Gula,16
Sereal,9


2. One-Hot Encoding Transaksi

In [5]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(transaksi).transform(transaksi)
df = pd.DataFrame(te_ary, columns=te.columns_)

In [6]:
display('Tabel One-Hot Encoding Transaksi:', df.head())

'Tabel One-Hot Encoding Transaksi:'

,Gula,Keju,Kopi,Mentega,Roti,Selai,Sereal,Susu,Teh,Telur
0,False,True,True,True,True,True,False,False,False,False
1,False,False,True,True,True,True,False,False,True,False
2,False,False,True,False,False,False,False,True,True,False
3,False,True,False,False,False,True,False,False,True,True
4,True,True,False,True,False,False,False,True,False,False


3. Frequent Itemset dengan Apriori

In [10]:
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
 freq = apriori(df, min_support=ms, use_colnames=True)
 print(f'min_support={ms}: {len(freq)} itemset ditemukan')

# Gunakan min_support yang menghasilkan jumlah itemset wajar (tidak 0, tidak ratusan)
freq_items = apriori(df, min_support=0.1, use_colnames=True)
freq_items = freq_items.sort_values('support', ascending=False)
print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


4. Bentuk & Saring Aturan Asosiasi

In [11]:
from mlxtend.frequent_patterns import association_rules

rules = association_rules(freq_items, metric='confidence',
 				min_threshold=0.5)
rules = rules[rules['lift'] > 1].sort_values('lift', ascending=False)

print(rules[['antecedents', 'consequents',
 		'support', 'confidence', 'lift']].head(10))

# Interpretasikan dalam sel Markdown:
# Aturan mana yang paling kuat (Lift tertinggi)?
# Apakah masuk akal secara bisnis (mis. Roti -> Selai)?

         antecedents consequents  support  confidence      lift
10       (Keju, Teh)     (Telur)     0.12    0.857143  2.380952
14  (Selai, Mentega)      (Kopi)     0.10    0.625000  1.953125
12      (Roti, Gula)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
9       (Telur, Teh)      (Keju)     0.12    0.600000  1.764706
13     (Selai, Kopi)   (Mentega)     0.10    0.714286  1.700680
8      (Telur, Keju)       (Teh)     0.12    0.750000  1.630435
11     (Selai, Gula)      (Roti)     0.10    0.500000  1.562500
15   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115


5. Rekomender Sederhana dengan Content-Based Filtering


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
 'produk': produk,
 'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
 		'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

In [15]:
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
 'produk': produk,
 'kategori': ['Bakery','Bakery','Dairy','Bakery','Dairy',
 		'Dairy','Minuman','Bumbu','Minuman','Dairy']
})

fitur = pd.get_dummies(katalog['kategori'])
sim_matrix = cosine_similarity(fitur)

def rekomendasi_serupa(nama_produk, top_n=3):
 idx = katalog.index[katalog['produk'] == nama_produk][0]
 skor = list(enumerate(sim_matrix[idx]))
 skor = sorted(skor, key=lambda x: x[1], reverse=True)
 skor = [s for s in skor if s[0] != idx][:top_n]
 return katalog.iloc[[i for i, _ in skor]]['produk'].tolist()

print('Mirip dengan Roti:', rekomendasi_serupa('Roti'))

Mirip dengan Roti: ['Selai', 'Sereal', 'Susu']


6. Bandingkan Kedua Pendekatan

In [16]:
produk_target = 'Roti'

# Dari association rules: cari consequents dari aturan yang antecedent-nya mengandung
produk_target
rules_terkait = rules[rules['antecedents'].apply(
 lambda x: produk_target in x)]
print('Rekomendasi dari Association Rules:')
print(rules_terkait[['consequents', 'lift']].head())

print('Rekomendasi dari Content-Based:', rekomendasi_serupa(produk_target))

# Diskusikan: apakah kedua pendekatan memberi rekomendasi yang konsisten?
# Kapan sebaiknya menggunakan salah satu, atau menggabungkan keduanya (hybrid)?

Rekomendasi dari Association Rules:
   consequents      lift
12     (Selai)  1.923077
1      (Selai)  1.322115
Rekomendasi dari Content-Based: ['Selai', 'Sereal', 'Susu']


#Kesimpulan:
1. Pembuatan Data Transaksi: Kita membuat data transaksi belanja fiktif dari 50 pelanggan. Setiap pelanggan membeli beberapa produk secara acak, dan kita bahkan menyisipkan 'pola rahasia' bahwa 'Roti' sering dibeli bersama 'Selai' untuk melihat apakah algoritma bisa menangkapnya.

2. Analisis Frekuensi Produk  Kita menghitung produk apa saja yang paling sering muncul dalam seluruh transaksi. Ini membantu kita memahami produk mana yang populer secara umum.

3. One-Hot Encoding : Kita mengubah daftar belanjaan menjadi format 'checklist' (tabel one-hot encoding). Ini seperti menandai produk apa saja yang ada di setiap transaksi, memudahkan komputer untuk menganalisisnya.

4. Mencari Itemset Sering Muncul dengan Apriori : Dengan data checklist tadi, kita menggunakan algoritma Apriori untuk menemukan kombinasi produk yang sering dibeli bersama (misalnya, 'Roti dan Selai'). Ini penting untuk menemukan pola pembelian.

5. Membentuk Aturan Asosiasi : Dari kombinasi produk yang sering muncul, kita membuat 'aturan jika-maka'. Contohnya, 'Jika pelanggan membeli Roti, kemungkinan besar ia juga membeli Selai'. Aturan ini sangat berguna untuk rekomendasi produk.

6. Rekomendasi Berbasis Konten (Content-Based Filtering) : Kita membuat katalog produk dengan kategori (misalnya, 'Roti' masuk kategori 'Bakery'). Kemudian, kita membangun sistem rekomendasi yang akan menyarankan produk yang 'serupa' berdasarkan kategori. Jadi, jika Anda suka 'Roti', sistem akan merekomendasikan 'Sereal' atau 'Selai' karena sama-sama kategori 'Bakery'.

7. Membandingkan Kedua Pendekatan: Terakhir, kita membandingkan hasil rekomendasi dari Aturan Asosiasi dan Rekomendasi Berbasis Konten untuk melihat apakah keduanya memberikan saran yang konsisten atau berbeda, dan kapan masing-masing pendekatan paling cocok digunakan. Misalnya, Aturan Asosiasi cocok untuk rekomendasi berdasarkan pola pembelian orang lain, sedangkan Content-Based cocok untuk merekomendasikan barang serupa yang mungkin Anda suka.